# 05 - Monte Carlo Simulation

Stress-test portfolios with thousands of simulated scenarios.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data import fetch_prices, compute_returns
from src.portfolio import build_portfolio_returns
from src.monte_carlo import (
    simulate_portfolio_monte_carlo,
    simulate_multivariate_monte_carlo,
    simulate_historical_bootstrap,
    monte_carlo_statistics,
    monte_carlo_return_stats,
    simulate_drawdown_cone,
)

In [ ]:
tickers = ["AAPL", "VWCE.MI", "BTC-USD"]
start_date = "2018-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)
print(f"Loaded {len(prices)} trading days")

## Define Portfolio

In [ ]:
weights = {"AAPL": 0.40, "VWCE.MI": 0.45, "BTC-USD": 0.15}
initial_value = 10000
n_simulations = 10000
n_days = 252

portfolio_returns = build_portfolio_returns(prices, weights)
print(f"Portfolio weights: {weights}")

## Parametric Monte Carlo (Normal Distribution)

In [ ]:
paths = simulate_portfolio_monte_carlo(
    portfolio_returns, n_simulations=n_simulations,
    n_days=n_days, initial_value=initial_value, random_seed=42,
)

n_show = 200
fig = go.Figure()
for i in range(min(n_show, paths.shape[1])):
    fig.add_trace(go.Scatter(
        y=paths[:, i], mode='lines',
        line=dict(width=0.5, color='rgba(100,100,255,0.1)'),
        showlegend=False,
    ))

mean_path = np.mean(paths, axis=1)
fig.add_trace(go.Scatter(y=mean_path, mode='lines', name='Mean', line=dict(width=3, color='red')))
fig.add_hline(y=initial_value, line_dash='dash', line_color='gray', annotation_text='Initial')

fig.update_layout(
    title=f'Parametric Monte Carlo ({n_simulations:,} simulations, {n_days} days)',
    yaxis_title='Portfolio Value ($)', xaxis_title='Trading Days',
    template='plotly_white', height=600,
)
fig.show()

## Historical Bootstrap Simulation

In [ ]:
bootstrap_paths = simulate_historical_bootstrap(
    portfolio_returns, n_simulations=n_simulations,
    n_days=n_days, initial_value=initial_value, random_seed=42,
)

n_show = 200
fig = go.Figure()
for i in range(min(n_show, bootstrap_paths.shape[1])):
    fig.add_trace(go.Scatter(
        y=bootstrap_paths[:, i], mode='lines',
        line=dict(width=0.5, color='rgba(100,200,100,0.1)'),
        showlegend=False,
    ))

mean_path = np.mean(bootstrap_paths, axis=1)
fig.add_trace(go.Scatter(y=mean_path, mode='lines', name='Mean', line=dict(width=3, color='red')))
fig.add_hline(y=initial_value, line_dash='dash', line_color='gray')

fig.update_layout(
    title=f'Historical Bootstrap ({n_simulations:,} simulations, {n_days} days)',
    yaxis_title='Portfolio Value ($)', xaxis_title='Trading Days',
    template='plotly_white', height=600,
)
fig.show()

## Simulation Statistics

In [ ]:
param_stats = monte_carlo_statistics(paths)
bootstrap_stats = monte_carlo_statistics(bootstrap_paths)

comparison = pd.DataFrame({"Parametric": param_stats, "Bootstrap": bootstrap_stats})
comparison.style.format('{:,.2f}')

In [ ]:
param_ret = monte_carlo_return_stats(paths, initial_value=initial_value)
bootstrap_ret = monte_carlo_return_stats(bootstrap_paths, initial_value=initial_value)

ret_comparison = pd.DataFrame({"Parametric": param_ret, "Bootstrap": bootstrap_ret})
ret_comparison.style.format('{:.4f}')

## Final Value Distribution

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Parametric', 'Bootstrap'))

fig.add_trace(go.Histogram(
    x=paths[-1, :], nbinsx=100, name='Parametric',
    marker_color='blue', opacity=0.7,
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=bootstrap_paths[-1, :], nbinsx=100, name='Bootstrap',
    marker_color='green', opacity=0.7,
), row=1, col=2)

for i in [1, 2]:
    fig.add_vline(x=initial_value, line_dash='dash', line_color='red', row=1, col=i)

fig.update_layout(
    title=f'Final Value Distribution ({n_days} days, {n_simulations:,} sims)',
    template='plotly_white', height=500,
)
fig.show()

## Drawdown Cone

Shows the range of drawdown paths at different quantiles.

In [ ]:
cone = simulate_drawdown_cone(
    portfolio_returns, n_simulations=2000,
    n_days=n_days, initial_value=initial_value, random_seed=42,
)

fig = go.Figure()
colors = {'q5': 'red', 'q25': 'orange', 'q50': 'blue', 'q75': 'orange', 'q95': 'green'}
for col in cone.columns:
    fig.add_trace(go.Scatter(
        y=cone[col], mode='lines', name=col,
        line=dict(width=2 if col in ['q50', 'mean'] else 1, color=colors.get(col, 'gray')),
    ))

fig.update_layout(
    title='Drawdown Cone (Quantile Paths)',
    yaxis_title='Portfolio Value ($)', xaxis_title='Trading Days',
    template='plotly_white', height=600,
)
fig.show()

## Compare Portfolios Under Stress

In [ ]:
portfolios_mc = {
    'Balanced (40/45/15)': {"AAPL": 0.40, "VWCE.MI": 0.45, "BTC-USD": 0.15},
    'Conservative (10/80/10)': {"AAPL": 0.10, "VWCE.MI": 0.80, "BTC-USD": 0.10},
    'Aggressive (30/20/50)': {"AAPL": 0.30, "VWCE.MI": 0.20, "BTC-USD": 0.50},
}

mc_results = {}
for name, w in portfolios_mc.items():
    ret = build_portfolio_returns(prices, w)
    sim = simulate_portfolio_monte_carlo(
        ret, n_simulations=5000, n_days=252, initial_value=10000, random_seed=42,
    )
    mc_results[name] = monte_carlo_return_stats(sim, initial_value=10000)

mc_df = pd.DataFrame(mc_results)
mc_df.style.format('{:.4f}')

In [ ]:
fig = go.Figure()
metrics_to_plot = ['Mean Return', 'Std Return', 'Avg Max Drawdown', 'Probability of Loss']
for metric in metrics_to_plot:
    fig.add_trace(go.Bar(
        x=list(mc_results.keys()),
        y=[mc_results[k][metric] for k in mc_results],
        name=metric,
    ))
fig.update_layout(
    title='Monte Carlo Comparison by Portfolio',
    barmode='group', template='plotly_white', height=500,
)
fig.show()